# preprocessing — phương án 3: chuyên khoa đơn nhãn

Gom văn bản gốc, giải quyết nhãn trên danh sách chuyên khoa trước lọc support. Mặc định lớp có ≥100 nhóm độc lập và mỗi holdout có ≥20 nhóm/lớp. Không giới hạn Top 5. Phạm vi thực nghiệm nằm trong configs/specialty_scope.json, không phải danh mục chuyên khoa chuẩn. Mặc định không cần WordNet.

Phạm vi hiện tại gồm đúng 26 nhãn do nhóm chỉ định; lọc support sau giải quyết nhãn.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "mtsamples.csv").exists():
    ROOT = ROOT.parent
if not (ROOT / "mtsamples.csv").exists():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks directory.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import json
import numpy as np
import pandas as pd
import joblib
from scipy.sparse import load_npz
from IPython.display import display, Image, Markdown
from src.config import PipelineConfig
from src.data_preprocessing import load_data
from scripts.run_preprocessing import run_pipeline

config = PipelineConfig()
OUT = config.output_dir
RERUN = False
if RERUN or not (OUT / "summary.json").exists():
    run_pipeline(config)
summary = json.loads((OUT / "summary.json").read_text(encoding="utf-8"))
display(pd.Series(summary["config"], name="Run configuration").to_frame())

## 1. Dữ liệu ban đầu và giá trị thiếu

Đầu vào mô hình là `transcription`; nhãn đích là `medical_specialty`.
Các cột văn bản khác chỉ dùng đối soát. Không loại bản ghi chỉ vì thiếu keywords.

In [ ]:
raw = load_data(config.input_path)
print(f"Rows: {len(raw):,}; labels: {raw.medical_specialty.nunique()}")
display(raw.head(3))
display(pd.read_csv(OUT / "tables/missing_values.csv"))
display(pd.read_csv(OUT / "tables/raw_length_statistics.csv"))

In [ ]:
def show_figures(*names):
    for name in names:
        display(Image(filename=str(OUT / "figures" / f"{name}.png"), width=1000))

show_figures("01_raw_class_distribution", "02_missing_values",
             "03_raw_text_lengths", "04_raw_length_by_class")

## 2. Giải quyết phạm vi nhãn và giữ dấu vết nguồn

Surgery + Urology → một bản Urology. Urology + Nephrology → rà soát, kể cả khi một nhãn bị lọc bởi support sau đó. Nhóm không có nhãn mục tiêu được lưu riêng. Va chạm cleaner khác nhãn được tách riêng, không coi là trùng bản gốc. Catalog giữ mọi dòng dùng được. Benchmark không còn nhóm trùng khác nhãn nhưng không phải gold standard.


In [ ]:
processed = pd.read_csv(OUT / "data/processed.csv")
removed = pd.read_csv(OUT / "tables/removed_rows.csv")
conflicts = pd.read_csv(OUT / "tables/conflicting_labels.csv")
display(pd.DataFrame({"raw_valid": summary["raw_duplicates_all_labels"],
                      "final": summary["final_duplicates"]}))
display(pd.read_csv(OUT / "tables/cleaning_flow.csv"))
display(removed.groupby("reason").size().rename("removed_rows").to_frame())
display(pd.read_csv(OUT / "tables/cleaning_examples.csv").head(4))
assert len(processed) + len(removed) == len(raw)
assert set(processed.source_row).isdisjoint(removed.source_row)
show_figures("05_cleaning_flow", "12_conflicting_labels")
decisions = pd.read_csv(OUT / "tables/specialty_group_decisions.csv")
resolved = pd.read_csv(OUT / "data/resolved_specialties.csv")
review = pd.read_csv(OUT / "tables/label_review_queue.csv")
display(decisions.groupby("decision").size().to_frame("raw_groups"))
display(review[["target_labels", "source_record_count", "text_excerpt"]].head())
assert len(decisions) == raw.loc[raw.transcription.notna()].transcription.map(
    __import__("src.data_preprocessing", fromlist=["normalized_text"]).normalized_text).nunique()
assert not processed.raw_group.duplicated().any()
assert not processed.label_ambiguous.any()
assert set(processed.source_row).issubset(set(resolved.source_row))
assert set(review.raw_group).isdisjoint(processed.raw_group)
assert decisions.loc[decisions.in_benchmark, "target_label_count"].eq(1).all()
show_figures("16_specialty_resolution", "17_specialty_support", "18_ambiguity_before_after")


## 3. Trực quan hóa trước và sau

So sánh ghép cặp trên cùng các bản ghi giữ lại. Làm sạch nhẹ có thể giữ nguyên hoặc tăng số token do tách dấu câu; giảm độ dài không phải mục tiêu chất lượng. Các lớp hiếm vẫn tồn tại trong catalog.

In [ ]:
display(pd.read_csv(OUT / "tables/class_counts_before_after.csv"))
display(pd.read_csv(OUT / "tables/paired_length_statistics.csv"))
show_figures("06_class_distribution_before_after", "07_paired_lengths",
             "08_clean_length_by_class", "09_frequent_words",
             "10_frequent_bigrams", "11_negation_retention")

## 4. Chia tập theo nhóm, có thể tái lập

Chia 5 fold theo nhóm có phân tầng: fold 0 test, fold 1 validation, phần còn lại train.
Tỷ lệ xấp xỉ 60/20/20; có thể lệch vì không tách các bản ghi trong cùng nhóm văn bản.
Kiểm tra cả hash văn bản gốc và hash văn bản sạch không giao nhau giữa các tập.

In [ ]:
splits = {name: pd.read_csv(OUT / "data" / f"{name}.csv")
          for name in ["train", "validation", "test"]}
display(pd.read_csv(OUT / "tables/split_summary.csv"))
display(pd.read_csv(OUT / "tables/split_class_counts.csv"))
for key in ["raw_group", "text_group"]:
    assert processed.groupby(key)["split"].nunique().max() == 1
assert not processed.duplicated(["raw_group", "medical_specialty"]).any()
show_figures("13_split_distribution")
for name in ["validation", "test"]:
    counts = splits[name].groupby("medical_specialty").text_group.nunique()
    assert counts.min() >= summary["config"]["min_holdout_groups"]


## 5. TF-IDF chỉ fit trên train và khám phá đặc trưng

Xuất cả unigram và unigram+bigram. Từ vựng và IDF chỉ được học từ train.
Các từ nổi bật bên dưới tính theo TF-IDF trung bình, không phải trọng số mô hình.
SVD là phép chiếu xuống hai chiều, không phải chỉ số đánh giá phân loại.

In [ ]:
for variant in ["unigram", "unigram_bigram"]:
    vectorizer = joblib.load(OUT / "artifacts" / f"tfidf_{variant}.joblib")
    for split, frame in splits.items():
        X = load_npz(OUT / "features" / f"X_{split}_{variant}.npz")
        y = np.load(OUT / "features" / f"y_{split}.npy")
        ids = np.load(OUT / "features" / f"source_rows_{split}.npy")
        assert X.shape[0] == len(frame) == len(y)
        np.testing.assert_array_equal(ids, frame.source_row.to_numpy())
        np.testing.assert_array_equal(y, frame.label_id.to_numpy())
        delta = X - vectorizer.transform(frame.clean_text)
        assert delta.nnz == 0 or abs(delta.data).max() < 1e-12
        print(variant, split, X.shape, f"sparsity={1-X.nnz/(X.shape[0]*X.shape[1]):.2%}")
show_figures("14_train_tfidf_terms", "15_train_svd_projection")

## 6. Bàn giao và giới hạn

- Chỉ dùng train.csv huấn luyện; catalog và resolved_specialties chứa cả holdout.
- CV dùng train_cv_folds.csv, fit TF-IDF và class weights trong mỗi fold.
- Nhóm nhiều nhãn mục tiêu nằm trong label_review_queue.csv. Không dùng lọc lớp hiếm để tự suy ra nhãn đúng.
- DL dùng transcription gốc và cùng split/label_id.
- Không còn xung đột nhãn không chứng minh nhãn đúng lâm sàng. Benchmark chỉ đại diện các ca đơn nhãn trong phạm vi, có thể dễ hơn dữ liệu gốc.
- Chưa xử lý trùng gần hoặc cách ly theo patient ID. Support ≥20 mỗi holdout không bảo đảm F1 ổn định.
- Không so sánh trực tiếp metric với v2 hoặc paper có dữ liệu/split khác.


In [ ]:
print("Report:", OUT / "report.html")
display(pd.Series({
    "raw_rows": summary["raw_rows"],
    "processed_rows": summary["processed_rows"],
    "unique_clean_texts": summary["final_duplicates"]["unique_text_groups"],
    "conflicting_groups": summary["final_duplicates"]["conflicting_label_groups"],
    "imbalance_ratio": summary["class_imbalance_ratio"],
}).to_frame("value"))
catalog = pd.read_csv(OUT / "data/cleaned_all.csv")
eligibility = pd.read_csv(OUT / "tables/class_eligibility.csv")
display(eligibility)
display(pd.read_csv(OUT / "tables/threshold_coverage.csv"))
assert len(catalog) == summary["catalog_rows"]
assert set(processed.source_row).issubset(set(catalog.source_row))
folds = pd.read_csv(OUT / "data/train_cv_folds.csv")
assert folds.groupby("text_group").cv_fold.nunique().max() == 1
assert pd.crosstab(folds.medical_specialty, folds.cv_fold).min().min() > 0
print("Catalog, class eligibility and training CV manifest verified.")

display(pd.read_csv(OUT / "tables/label_ambiguity_by_split.csv"))
print("Empirical deterministic clean-text accuracy ceiling:", summary["deterministic_clean_text_accuracy_ceiling"])
